# Task 3.3 — Security Staffing Recommendation Table

Using 30-day rolling average from `silver.zone_occupancy_15min` per (`day_of_week`, `time_slot`):
- Join with today's flight schedule: estimate passenger wave (departure − 90 min = check-in peak)
- `predicted_queue_length` per security zone per 15-minute slot
- `recommended_lanes_open`: predicted_queue / lanes_capacity_per_hour × 15 min factor
- Write to `gold.security_staffing_recommendation` — used by terminal operations manager


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"
GOLD_DIR = "/FileStore/delta_lake/gold"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_3_3_Security_Staffing")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Silver Zone Occupancy & Flight Schedule

In [ ]:
# ── Read required tables ──────────────────────────────────────────
zone_df = spark.read.format("delta").load(f"{SILVER_DIR}/zone_occupancy_15min")
flight_ops_df = spark.read.format("delta").load(f"{SILVER_DIR}/flight_operations")

# Filter to SECURITY_LANE zones only
security_zones_df = zone_df.filter(F.col("location_type") == "SECURITY_LANE")

print(f"Security zone 15-min records: {security_zones_df.count()}")
print(f"Flight operations: {flight_ops_df.count()}")
security_zones_df.show(5, truncate=False)


## Step 2 — 30-Day Rolling Average by Day-of-Week + Time Slot

In [ ]:
# ── Historical averages per (day_of_week, time_slot, zone) ────────
# Extract day_of_week and time_slot (hour:minute)
hist_df = (security_zones_df
    .withColumn("day_of_week", F.dayofweek("time_bucket"))
    .withColumn("time_slot", F.date_format("time_bucket", "HH:mm"))
)

# 30-day rolling average (using all available data as proxy)
avg_by_pattern_df = (hist_df
    .groupBy("zone_id", "terminal", "day_of_week", "time_slot")
    .agg(
        F.round(F.avg("avg_queue_length"), 1).alias("hist_avg_queue"),
        F.round(F.avg("avg_wait_time_mins"), 1).alias("hist_avg_wait_mins"),
        F.round(F.avg("cumulative_pax_through"), 0).alias("hist_avg_pax"),
        F.count("*").alias("data_points")
    )
)

print(f"Historical pattern records: {avg_by_pattern_df.count()}")
avg_by_pattern_df.show(5, truncate=False)


## Step 3 — Estimate Passenger Wave from Flight Schedule

In [ ]:
# ── Passenger wave estimation ─────────────────────────────────────
# Check-in peak arrives ~90 minutes before departure
# Security peak ~60 minutes before departure
pax_wave_df = (flight_ops_df
    .withColumn("security_peak_ts",
        F.from_unixtime(
            F.unix_timestamp("scheduled_departure") - 60 * 60  # 60 min before
        )
    )
    .withColumn("security_peak_slot",
        F.date_format("security_peak_ts", "HH:mm"))
    .withColumn("departure_day_of_week",
        F.dayofweek("scheduled_departure"))
    .groupBy("terminal", "departure_day_of_week", "security_peak_slot")
    .agg(
        F.sum("pax_count_booked").alias("estimated_pax_wave"),
        F.count("*").alias("departing_flights")
    )
)

print(f"Passenger wave patterns: {pax_wave_df.count()}")
pax_wave_df.orderBy("estimated_pax_wave", ascending=False).show(10, truncate=False)


## Step 4 — Compute Predicted Queue & Recommended Lanes

In [ ]:
# ── Predicted queue and staffing recommendation ──────────────────
LANES_CAPACITY_PER_HOUR = 150  # passengers per lane per hour
FIFTEEN_MIN_FACTOR = LANES_CAPACITY_PER_HOUR / 4  # per 15-min slot

# Join historical patterns with passenger wave
staffing_df = (avg_by_pattern_df
    .join(pax_wave_df,
        (avg_by_pattern_df.terminal == pax_wave_df.terminal) &
        (avg_by_pattern_df.day_of_week == pax_wave_df.departure_day_of_week) &
        (avg_by_pattern_df.time_slot == pax_wave_df.security_peak_slot),
        "left")
    .drop(pax_wave_df.terminal)
)

# Predicted queue = historical average + passenger wave factor
staffing_df = staffing_df.withColumn(
    "predicted_queue_length",
    F.round(
        F.col("hist_avg_queue") +
        F.coalesce(F.col("estimated_pax_wave"), F.lit(0)) * 0.1,  # 10% security factor
        0
    ).cast("int")
)

# Recommended lanes open
staffing_df = staffing_df.withColumn(
    "recommended_lanes_open",
    F.ceil(F.col("predicted_queue_length") / F.lit(FIFTEEN_MIN_FACTOR))
)

# Confidence level based on data points
staffing_df = staffing_df.withColumn(
    "confidence_level",
    F.when(F.col("data_points") >= 20, "HIGH")
     .when(F.col("data_points") >= 10, "MEDIUM")
     .otherwise("LOW")
)

staffing_df = staffing_df.withColumn("processing_ts", F.current_timestamp())

print("\nStaffing Recommendation Sample:")
staffing_df.select(
    "zone_id", "terminal", "day_of_week", "time_slot",
    "predicted_queue_length", "recommended_lanes_open", "confidence_level"
).orderBy("predicted_queue_length", ascending=False).show(10, truncate=False)


## Step 5 — Write to Gold Delta

In [ ]:
# ── Write gold.security_staffing_recommendation ──────────────────
gold_security_path = f"{GOLD_DIR}/security_staffing_recommendation"

(staffing_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_security_path))

print(f"Written to {gold_security_path}")
result_df = spark.read.format("delta").load(gold_security_path)
print(f"Gold security_staffing_recommendation total: {result_df.count()}")

print("\nRecommendations by Confidence Level:")
result_df.groupBy("confidence_level").agg(
    F.count("*").alias("records"),
    F.round(F.avg("recommended_lanes_open"), 1).alias("avg_lanes")
).show()
